# SmartPantry AI | Role 5: Recipe Matching and Ranking

**Phase 3 demonstration notebook** based on the team’s `modules/recipe_recommender.py` provided for review. This notebook demonstrates the existing deterministic baseline with mock data. **It does not call Spoonacular or OpenAI, and it does not train an AI model.** The AI assistant and live API integration are planned Phase 3 extensions.

## 1. How to run
Run the cells from top to bottom in Google Colab or Jupyter. No API key, external file, or additional Python package is required. The functions below are copied from the supplied module so this notebook is independently runnable; future changes should be synchronized with the module.

## 2. Baseline functions from `recipe_recommender.py`
The following function implementations and ingredient aliases are taken from the supplied Python file. Diagnostic `print` statements inside the expiration function are preserved for transparency.

In [ ]:
def normalize_ingredient_name(name):
    cleaned_name = name.strip().lower()

    return cleaned_name

ingredient_aliases = {
    "baby spinach": "spinach",
    "chicken breasts": "chicken breast",
    "brown rice": "rice",
    "basmati rice": "rice",
    "regular rice": "rice",
    "jasmine rice": "rice"
}

def ingredients_match(pantry_name, recipe_name):
    cleaned_pantry_name = normalize_ingredient_name(pantry_name)
    cleaned_recipe_name = normalize_ingredient_name(recipe_name)

    if cleaned_pantry_name == cleaned_recipe_name:
        return True

    if cleaned_recipe_name in ingredient_aliases:
        alias_name = ingredient_aliases[cleaned_recipe_name]

        if alias_name == cleaned_pantry_name:
            return True

    return False

def get_pantry_names(pantry):
    pantry_names = []

    for item in pantry:
        pantry_names.append(item["name"])
    return pantry_names

def get_expiration_points(days_until_expiration):
    if days_until_expiration <= 1:
        return 3
    elif days_until_expiration <= 3:
        return 2
    else:
        return 0

def calculate_total_expiration_points(pantry):
    total_points = 0

    for item in pantry:
        total_points = total_points + get_expiration_points(item["days_until_expiration"])

    return total_points

def check_recipe_ingredients(recipe, pantry_names):
    available_count = 0 
    missing_count = 0

    for ingredient in recipe["ingredients"]:
        ingredient_found = False

        for pantry_name in pantry_names:
            if ingredients_match(pantry_name, ingredient):
                ingredient_found = True  
                break      

        if ingredient_found == True:
            available_count = available_count + 1
        else:
            missing_count = missing_count + 1

    return available_count, missing_count

def calculate_recipe_expiration_score(recipe, pantry):
    expiration_score = 0

    for ingredient in recipe["ingredients"]:
        for item in pantry:
            match_result = ingredients_match(
                item["name"],
                ingredient
            )

            print(
                "MATCH RESULT:",
                repr(item["name"]),
                repr(ingredient),
                match_result
            )

            if match_result:
                print(
                    "EXPIRATION MATCH FOUND:",
                    item["name"],
                    ingredient
                )

                expiration_score = expiration_score + get_expiration_points(
                    item["days_until_expiration"]
                )
                       
    return expiration_score

def calculate_pantry_coverage(available_count, total_ingredients):
    if total_ingredients > 0:
        pantry_coverage = (available_count / total_ingredients) * 100
    else:
        pantry_coverage = 0
  
    return pantry_coverage

def calculate_expiration_coverage(expiration_score, total_expiration_points):
    if total_expiration_points > 0:
        expiration_coverage = (expiration_score / total_expiration_points) * 100
    else:
        expiration_coverage = 0

    return expiration_coverage

def calculate_final_score(
    pantry_coverage,
    expiration_coverage,
    pantry_weight,
    expiration_weight
):
    final_score = (
        pantry_coverage * pantry_weight
        + expiration_coverage * expiration_weight
    )

    return final_score

def score_recipe(
        recipe,
        pantry,
        pantry_weight,
        expiration_weight
):
    pantry_names = get_pantry_names(pantry)

    available_count, missing_count = check_recipe_ingredients(
        recipe,
        pantry_names
    )

    expiration_score = calculate_recipe_expiration_score(
        recipe,
        pantry
    )

    total_ingredients = len(recipe["ingredients"])

    pantry_coverage = calculate_pantry_coverage(
        available_count,
        total_ingredients
    )

    total_expiration_points = calculate_total_expiration_points(pantry)

    expiration_coverage = calculate_expiration_coverage(
        expiration_score,
        total_expiration_points
    )

    final_score = calculate_final_score(
        pantry_coverage,
        expiration_coverage,
        pantry_weight,
        expiration_weight
    )

    return (
        available_count,
        missing_count,
        expiration_score,
        expiration_coverage,
        final_score,
        pantry_coverage
    )

def rank_recipes (
    recipes,
    pantry,
    pantry_weight,
    expiration_weight
):
    recipe_results = []

    for recipe in recipes:
        (
            available_count,
            missing_count,
            expiration_score,
            expiration_coverage,
            final_score,
            pantry_coverage
        ) = score_recipe(
            recipe,
            pantry,
            pantry_weight,
            expiration_weight
        )

        recipe_result = {
            "name": recipe["name"],
            "score": final_score
        }

        recipe_results.append(recipe_result)

    recipe_results.sort(
        key=lambda result: result["score"],
        reverse=True
    )

    return recipe_results

## 3. Sample pantry and recipe candidates
The same sample data as the supplied module are used. These are mock recipes, not Spoonacular results.

In [ ]:
pantry = [
    {"name": "chicken breast", "quantity": 2, "unit": "pieces", "days_until_expiration": 2},
    {"name": "spinach", "quantity": 1, "unit": "bag", "days_until_expiration": 1},
    {"name": "rice", "quantity": 2, "unit": "cups", "days_until_expiration": 30},
]
recipes = [
    {"name": "Chicken Rice Bowl", "ingredients": ["chicken breast", "rice"]},
    {"name": "Chicken Spinach Rice Bowl", "ingredients": ["chicken breast", "spinach", "rice", "garlic"]},
]
pantry_weight = 0.60
expiration_weight = 0.40
print("Pantry items:", len(pantry), "| Recipe candidates:", len(recipes))

## 4. Ingredient normalization and matching
The current matching logic normalizes case and whitespace and supports aliases when the **recipe ingredient** uses the alias. A pantry-side alias is not yet handled symmetrically.

In [ ]:
assert normalize_ingredient_name(" Baby Spinach ") == "baby spinach"
assert ingredients_match("spinach", "baby spinach") is True
assert ingredients_match("rice", "brown rice") is True
assert ingredients_match("rice", "rice vinegar") is False
print("Normalization and baseline alias tests passed.")
print("Reverse alias currently matches:", ingredients_match("baby spinach", "spinach"))

## 5. Rank recipes with the existing 60% / 40% formula
Pantry coverage measures the fraction of recipe ingredients already available. Expiration coverage measures the fraction of the pantry’s expiration-priority points used by the recipe. Higher final scores rank first.

In [ ]:
from contextlib import redirect_stdout
from io import StringIO

# Suppress existing debugging messages only in this demonstration cell.
with redirect_stdout(StringIO()):
    ranked = rank_recipes(recipes, pantry, pantry_weight, expiration_weight)
for position, result in enumerate(ranked, 1):
    print(f"{position}. {result['name']}: {result['score']:.1f}")
assert ranked[0]["name"] == "Chicken Spinach Rice Bowl"
assert abs(ranked[0]["score"] - 85.0) < 1e-9
assert abs(ranked[1]["score"] - 76.0) < 1e-9
print("Baseline ranking tests passed.")

## 6. Examine individual scores

In [ ]:
for recipe in recipes:
    with redirect_stdout(StringIO()):
        available, missing, exp_points, exp_coverage, final, pantry_coverage = score_recipe(
            recipe, pantry, pantry_weight, expiration_weight
        )
    print(f"{recipe['name']}: {available} available, {missing} missing, "
          f"pantry coverage {pantry_coverage:.1f}%, expiration coverage {exp_coverage:.1f}%, "
          f"final score {final:.1f}")

## 7. Known limitations and Phase 3 next steps

- **Live Spoonacular:** The supplied module sets `use_live_api = False` but reads `response` outside that conditional, causing an error on import/run. Refactor API calls behind a function or a guarded block before importing it directly.
- **Aliases:** Matching is one-directional; add tests and a symmetric normalization strategy.
- **Test typo:** The source’s `"baby spianch"` is misspelled and does not match the existing alias dictionary.
- **Debugging:** Remove or configure verbose prints when moving to production.
- **Real recipe validation:** Spoonacular ingredient payloads will need conversion into the `{"name", "ingredients"}` shape used here.
- **AI assistant (planned):** Add natural-language pantry entry and recipe adaptation using the approved pretrained model. Label any simulated AI outputs as mock; evaluate live outputs separately.
- **Evaluation:** Add more diverse pantry/recipe cases, boundary tests, and evidence of integration results in Weeks 8–10.